<a href="https://colab.research.google.com/github/Sprg72/Data-Engineer/blob/main/notebooks/Pyspark_lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# step1
!pip install findspark pyspark


In [2]:
# step2
import findspark
findspark.init()


In [3]:
#step3
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('myapp').getOrCreate()

In [4]:
sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=myapp>

In [6]:
# union --> to concatenate elements of 2 RDDs.
data1 = [(101,'amar'), (102,'siva')]
r1 = sc.parallelize(data1)
data2 = [(201,'sri'), (202,'hamsini')]
r2 = sc.parallelize(data2)


In [7]:
r = r1.union(r2)
r.collect()

[(101, 'amar'), (102, 'siva'), (201, 'sri'), (202, 'hamsini')]

In [8]:
data3 = [(301,'laasya'), (302,'spgreddy')]
r3 = sc.parallelize(data3)
r = r1.union(r2).union(r3)
r.collect()

[(101, 'amar'),
 (102, 'siva'),
 (201, 'sri'),
 (202, 'hamsini'),
 (301, 'laasya'),
 (302, 'spgreddy')]



```
# input file1: emp1.txt

101, amar,90000,m,11
102, amala,20000,f,12
103, ankit,40000,m,13
104, ankita,60000,f,13
105, anusha,110000,f,12
106, anuz,20000,m,11
107, akash,100000,m,12
108, siva,20000,m,14
109, sivani,30000,f,15
110, mani,30000,m,12
111, manisha,300000,f,13
112, sivam,200000,m,12
113, varun,200000,m,13

schema ---> id,name,salary,gender,dno

input file2: emp2.txt

201,rama,90000,m,11
202,amba,200000,f,12
203,ankush,20000,m,13
204,rajeshwari,60000,f,13


schema ---> id,name,salary,gender,dno


input file3: emp3.txt

301,aaaa,11,m,30000,25
302,bbbb,12,f,40000,22
303,cccc,13,m,50000,21

schema ---> id,name,dno,gender,salary,age

input file4: emp4.txt

401,xxxx,12,m,hyd,90000
402,yyyyy,13,f,pune,100000
403,zzzz,14,f,delhi,200000
404,uuuu,11,m,hyd,400000

schema --> id,name,dno,gender,city,salary

expected output schema:
    --> id,name,salary,gender,dno,age,city



emp1 = sc.textFile('/content/emp1.txt') #id,name,salary,gender,dno
emp2 = sc.textFile('/content/emp2.txt') #id,name,salary,gender,dno
emp3 = sc.textFile('/content/emp3.txt') #id,name,dno,gender,salary,age
emp4 = sc.textFile('/content/emp4.txt') #id,name,dno,gender,city,salary



def toemp1andemp2(line):
  line += ',0,NoCity'
  return line

def toemp3(line):
  w = line.strip().lower().split(',')
  # expected = [id, name, salary, gender, dno, age, city]
    expected = [w[0], w[1], w[-2], w[3], w[2], w[-1], 'NoCity']
    outline = ','.join(expected)
    return outline

def toemp4(line):
    w = line.strip().lower().split(',')
    # expected = [id, name, salary, gender, dno, age, city]
    expected = [w[0], w[1], w[-1], w[3], w[2], '0', w[-2]]
    outline = ','.join(expected)
    return outline


```




In [9]:
emp1 = sc.textFile('/content/emp1.txt') #id,name,salary,gender,dno
emp2 = sc.textFile('/content/emp2.txt') #id,name,salary,gender,dno
emp3 = sc.textFile('/content/emp3.txt') #id,name,dno,gender,salary,age
emp4 = sc.textFile('/content/emp4.txt') #id,name,dno,gender,city,salary

In [10]:
def toemp1andemp2(line):
  line += ',0,NoCity'
  return line

In [11]:
e1 = emp1.map(toemp1andemp2)
e1.collect()

['101, amar,90000,m,11,0,NoCity',
 '102, amala,20000,f,12,0,NoCity',
 '103, ankit,40000,m,13,0,NoCity',
 '104, ankita,60000,f,13,0,NoCity',
 '105, anusha,110000,f,12,0,NoCity',
 '106, anuz,20000,m,11,0,NoCity',
 '107, akash,100000,m,12,0,NoCity',
 '108, siva,20000,m,14,0,NoCity',
 '109, sivani,30000,f,15,0,NoCity',
 '110, mani,30000,m,12,0,NoCity',
 '111, manisha,300000,f,13,0,NoCity',
 '112, sivam,200000,m,12,0,NoCity',
 '113, varun,200000,m,13,0,NoCity']

In [12]:
e2 = emp2.map(toemp1andemp2)
e2.collect()

['201,rama,90000,m,11,0,NoCity',
 '202,amba,200000,f,12,0,NoCity',
 '203,ankush,20000,m,13,0,NoCity',
 '204,rajeshwari,60000,f,13,0,NoCity']

In [14]:
def toemp3(line):
  w = line.strip().lower().split(',')
  # expected = [id, name, salary, gender, dno, age, city]
  expected = [w[0], w[1], w[-2], w[3], w[2], w[-1], 'NoCity']
  outline = ','.join(expected)
  return outline

In [16]:
e3 = emp3.map(toemp3)
e3.collect()

['301,aaaa,30000,m,11,25,NoCity',
 '302,bbbb,40000,f,12,22,NoCity',
 '303,cccc,50000,m,13,21,NoCity']

In [17]:
def toemp4(line):
    w = line.strip().lower().split(',')
    # expected = [id, name, salary, gender, dno, age, city]
    expected = [w[0], w[1], w[-1], w[3], w[2], '0', w[-2]]
    outline = ','.join(expected)
    return outline

In [18]:
e4 = emp4.map(toemp4)
e4.collect()

['401,xxxx,90000,m,12,0,hyd',
 '402,yyyyy,100000,f,13,0,pune',
 '403,zzzz,200000,f,14,0,delhi',
 '404,uuuu,400000,m,11,0,hyd']

In [21]:
# way1
# e = emp1.union(emp2).union(emp3.union(emp4))
# e = e1.union(e2).union(e3.union(e4))
# way 2
e = e1 + e2 + e3 + e4
e.count()
e.collect()

['101, amar,90000,m,11,0,NoCity',
 '102, amala,20000,f,12,0,NoCity',
 '103, ankit,40000,m,13,0,NoCity',
 '104, ankita,60000,f,13,0,NoCity',
 '105, anusha,110000,f,12,0,NoCity',
 '106, anuz,20000,m,11,0,NoCity',
 '107, akash,100000,m,12,0,NoCity',
 '108, siva,20000,m,14,0,NoCity',
 '109, sivani,30000,f,15,0,NoCity',
 '110, mani,30000,m,12,0,NoCity',
 '111, manisha,300000,f,13,0,NoCity',
 '112, sivam,200000,m,12,0,NoCity',
 '113, varun,200000,m,13,0,NoCity',
 '201,rama,90000,m,11,0,NoCity',
 '202,amba,200000,f,12,0,NoCity',
 '203,ankush,20000,m,13,0,NoCity',
 '204,rajeshwari,60000,f,13,0,NoCity',
 '301,aaaa,30000,m,11,25,NoCity',
 '302,bbbb,40000,f,12,22,NoCity',
 '303,cccc,50000,m,13,21,NoCity',
 '401,xxxx,90000,m,12,0,hyd',
 '402,yyyyy,100000,f,13,0,pune',
 '403,zzzz,200000,f,14,0,delhi',
 '404,uuuu,400000,m,11,0,hyd']

In [22]:
e.getNumPartitions()

8

In [23]:
e.coalesce(1).saveAsTextFile('/content/allemps')



```
# output file part-00000 contains the below output

101, amar,90000,m,11,0,NoCity
102, amala,20000,f,12,0,NoCity
103, ankit,40000,m,13,0,NoCity
104, ankita,60000,f,13,0,NoCity
105, anusha,110000,f,12,0,NoCity
106, anuz,20000,m,11,0,NoCity
107, akash,100000,m,12,0,NoCity
108, siva,20000,m,14,0,NoCity
109, sivani,30000,f,15,0,NoCity
110, mani,30000,m,12,0,NoCity
111, manisha,300000,f,13,0,NoCity
112, sivam,200000,m,12,0,NoCity
113, varun,200000,m,13,0,NoCity
201,rama,90000,m,11,0,NoCity
202,amba,200000,f,12,0,NoCity
203,ankush,20000,m,13,0,NoCity
204,rajeshwari,60000,f,13,0,NoCity
301,aaaa,30000,m,11,25,NoCity
302,bbbb,40000,f,12,22,NoCity
303,cccc,50000,m,13,21,NoCity
401,xxxx,90000,m,12,0,hyd
402,yyyyy,100000,f,13,0,pune
403,zzzz,200000,f,14,0,delhi
404,uuuu,400000,m,11,0,hyd

```



In [24]:
emp3.collect()

['301,aaaa,11,m,30000,25', '302,bbbb,12,f,40000,22', '303,cccc,13,m,50000,21']

In [27]:
words3 = emp3.map(lambda x : x.split(','))
words3.collect()

[['301', 'aaaa', '11', 'm', '30000', '25'],
 ['302', 'bbbb', '12', 'f', '40000', '22'],
 ['303', 'cccc', '13', 'm', '50000', '21']]

In [29]:
e3 = words3.map(lambda x: [x[0],x[1],x[-2],x[3],x[2],x[-1], 'NoCity'])
e3.collect()

[['301', 'aaaa', '30000', 'm', '11', '25', 'NoCity'],
 ['302', 'bbbb', '40000', 'f', '12', '22', 'NoCity'],
 ['303', 'cccc', '50000', 'm', '13', '21', 'NoCity']]

In [30]:
e3 = e3.map(lambda x : ','.join(x))
e3.collect()

['301,aaaa,30000,m,11,25,NoCity',
 '302,bbbb,40000,f,12,22,NoCity',
 '303,cccc,50000,m,13,21,NoCity']

In [32]:
words1 = e1.map(lambda x : x.split(','))
branch1 = words1.map(lambda x : ('Branch1', int(x[2])))
words2 = e2.map(lambda x : x.split(','))
branch2 = words2.map(lambda x : ('Branch2', int(x[2])))
words3 = e3.map(lambda x : x.split(','))
branch3 = words3.map(lambda x : ('Branch3', int(x[2])))
words4 = e4.map(lambda x : x.split(','))
branch4 = words4.map(lambda x : ('Branch4', int(x[2])))

In [35]:
branch1.collect()

[('Branch1', 90000),
 ('Branch1', 20000),
 ('Branch1', 40000),
 ('Branch1', 60000),
 ('Branch1', 110000),
 ('Branch1', 20000),
 ('Branch1', 100000),
 ('Branch1', 20000),
 ('Branch1', 30000),
 ('Branch1', 30000),
 ('Branch1', 300000),
 ('Branch1', 200000),
 ('Branch1', 200000)]

In [34]:
branch2.collect()

[('Branch2', 90000),
 ('Branch2', 200000),
 ('Branch2', 20000),
 ('Branch2', 60000)]

In [36]:
branch3.collect()

[('Branch3', 30000), ('Branch3', 40000), ('Branch3', 50000)]

In [37]:
branch4.collect()

[('Branch4', 90000),
 ('Branch4', 100000),
 ('Branch4', 200000),
 ('Branch4', 400000)]

In [38]:
allbranches = branch1 + branch2 + branch3 + branch4
allbranches.collect()

[('Branch1', 90000),
 ('Branch1', 20000),
 ('Branch1', 40000),
 ('Branch1', 60000),
 ('Branch1', 110000),
 ('Branch1', 20000),
 ('Branch1', 100000),
 ('Branch1', 20000),
 ('Branch1', 30000),
 ('Branch1', 30000),
 ('Branch1', 300000),
 ('Branch1', 200000),
 ('Branch1', 200000),
 ('Branch2', 90000),
 ('Branch2', 200000),
 ('Branch2', 20000),
 ('Branch2', 60000),
 ('Branch3', 30000),
 ('Branch3', 40000),
 ('Branch3', 50000),
 ('Branch4', 90000),
 ('Branch4', 100000),
 ('Branch4', 200000),
 ('Branch4', 400000)]

In [39]:
branchtot = allbranches.reduceByKey(lambda x, y : x+y)
branchtot.collect()

[('Branch3', 120000),
 ('Branch2', 370000),
 ('Branch1', 1220000),
 ('Branch4', 790000)]